In [1]:
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.sql import SparkSession

# Stop any existing session first
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("TaxiPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [2]:
df = spark.read.parquet("../output/cleaned/")

In [3]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|pickup_year|pickup_month|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-----------+------------+
|       1| 2015-01-19 14:40:48|  2015-01-19 15:01:36|              1|          3.4|  

- Hourly demand — trip count by hour of day 
- Revenue by vendor — average fare, average tip, total trips per VendorID

In [14]:
from pyspark.sql.functions import round, avg, count, sum
from pyspark.sql.functions import format_number

revenue_by_vendor = df.groupBy("VendorID") \
    .agg(
        format_number(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip"),
        count("*").alias("total_trips")
    )

In [15]:
revenue_by_vendor.show()

+--------+--------------+--------+-------+-----------+
|VendorID| total_revenue|avg_fare|avg_tip|total_trips|
+--------+--------------+--------+-------+-----------+
|       1|325,071,176.92|   12.13|   1.68|   21405119|
|       2|379,670,977.77|   12.33|   1.71|   24602198|
+--------+--------------+--------+-------+-----------+



In [ ]:
from pyspark.sql import functions as F

hourly_demand = (
    df.groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour"))
      .count()
      .orderBy("pickup_hour")
)

In [13]:
hourly_demand.show(24)

+-----------+-------+
|pickup_hour|  count|
+-----------+-------+
|          0|1646682|
|          1|1210880|
|          2| 890548|
|          3| 666208|
|          4| 494662|
|          5| 459876|
|          6|1004815|
|          7|1715946|
|          8|2107013|
|          9|2124952|
|         10|2058635|
|         11|2139882|
|         12|2272923|
|         13|2269843|
|         14|2368876|
|         15|2329427|
|         16|2077196|
|         17|2440413|
|         18|2899057|
|         19|2892479|
|         20|2683639|
|         21|2621708|
|         22|2505162|
|         23|2126495|
+-----------+-------+



In [16]:
hourly_demand.write \
    .mode("overwrite") \
    .parquet("../output/aggregations/hourly_demand/")

revenue_by_vendor.write \
    .mode("overwrite") \
    .parquet("../output/aggregations/revenue_by_vendor/")

In [18]:
hourly_demand_df = spark.read.parquet("../output/aggregations/hourly_demand/")
revenue_by_vendor_df = spark.read.parquet("../output/aggregations/revenue_by_vendor/")

In [21]:
hourly_demand_df.show(24)

+-----------+-------+
|pickup_hour|  count|
+-----------+-------+
|          0|1646682|
|          1|1210880|
|          2| 890548|
|          3| 666208|
|          4| 494662|
|          5| 459876|
|          6|1004815|
|          7|1715946|
|          8|2107013|
|          9|2124952|
|         10|2058635|
|         11|2139882|
|         12|2272923|
|         13|2269843|
|         14|2368876|
|         15|2329427|
|         16|2077196|
|         17|2440413|
|         18|2899057|
|         19|2892479|
|         20|2683639|
|         21|2621708|
|         22|2505162|
|         23|2126495|
+-----------+-------+



In [22]:
revenue_by_vendor_df.show()

+--------+--------------+--------+-------+-----------+
|VendorID| total_revenue|avg_fare|avg_tip|total_trips|
+--------+--------------+--------+-------+-----------+
|       1|325,071,176.92|   12.13|   1.68|   21405119|
|       2|379,670,977.77|   12.33|   1.71|   24602198|
+--------+--------------+--------+-------+-----------+



duration_minutes = (unix_timestamp(dropoff) - unix_timestamp(pickup)) / 60

In [34]:
from pyspark.sql.functions import unix_timestamp

duration_minutes  = df.withColumn("trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime")
    - unix_timestamp("tpep_pickup_datetime")
    ) / 60
    )

duration_minutes = duration_minutes.filter(col("trip_duration_minutes") > 0)

In [35]:
from pyspark.sql import functions as F

avg_duration_minutes_by_hour = (
    duration_minutes
    .groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour"))
    .agg(round(avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"))
    .orderBy("pickup_hour")
)

In [36]:
avg_duration_minutes_by_hour.show(24)

+-----------+--------------------+
|pickup_hour|avg_duration_minutes|
+-----------+--------------------+
|          0|               14.96|
|          1|               14.69|
|          2|               14.36|
|          3|               14.72|
|          4|               14.91|
|          5|                13.8|
|          6|               12.88|
|          7|               13.54|
|          8|               14.83|
|          9|               14.75|
|         10|                14.8|
|         11|               15.01|
|         12|               15.13|
|         13|               15.33|
|         14|               16.17|
|         15|               16.57|
|         16|               15.79|
|         17|               15.55|
|         18|               14.78|
|         19|               13.92|
|         20|               13.96|
|         21|               14.04|
|         22|               16.67|
|         23|                14.8|
+-----------+--------------------+



In [37]:

avg_duration_minutes_by_hour.write \
    .mode("overwrite") \
    .parquet("../output/aggregations/avg_duration_by_hour/")